# TESTE — Pair-Aware BioBERTpt-all **sem pesos de classe** (`class_weight="none"`) — RECLin-PT (Colab T4)

**Execução exploratória de teste.** Repete a Pair-Aware linear oficial de `notebooks/reclin_pair_aware.ipynb` (`pucpr/biobertpt-all`, seed 42) mudando **um único parâmetro**: `class_weight="balanced"` → `class_weight="none"`. Todo o resto é idêntico ao protocolo (D10): mesma estratégia `pair_aware`, mesma cabeça linear `[h_cls ; h_E1 ; h_E2] → Dropout → Linear(3d, 3)`, mesmas partições v2, mesma revisão dos pesos, `max_gap=25`, 3 épocas, `batch_size=64`, `lr=2e-5`, seleção da época pelo **macro-F1 do DEV completo**.

**Por quê.** A auditoria de 05/11/2026 mostrou que os pesos `balanced` dão a `negation_of` peso efetivo ≈ 61× o de `no_relation` (≈ 115× nominal), o que empurra o modelo para recall alto e precisão baixa (DEV oficial: P 0,742 / R 0,940 / F1 0,829). A correção pós-hoc ÷c subiu a precisão para 0,788 e o recall caiu para 0,893. Este teste verifica se **treinar sem pesos** leva a um ponto de operação mais preciso diretamente no treino.

**Natureza.** Como `PARAMETROS` difere do protocolo, a execução é **EXPLORATÓRIA**: recebe pasta própria (`pair_aware__<config_id>__t1`), **nunca** usa nem sobrescreve a pasta oficial `pairaware_biobertpt_seed42`, não é elegível ao TEST e não entra na fase científica sem decisão explícita do autor (emenda E3, registro de tentativas).

**Só DEV.** Este notebook **não lê nem avalia o TEST**. A seção 8.3 compara com a Pair-Aware oficial e com o baseline no **DEV**, sobre o mesmo `y_true`.

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre os secrets `GITHUB_PAT` e `HF_TOKEN` (ícone de chave, com **Notebook access** ligado). O treino é espelhado a cada checkpoint no Google Drive (`MyDrive/RECLin-PT/execucoes/`); se o runtime cair, **rode as seções 1–7 de novo**: a seção 7 **retoma** a execução interrompida desta mesma configuração.

Envios para fora (GitHub e Hugging Face) só acontecem nas seções 10 e 11, e só depois de você mudar a confirmação explícita para `True`.

## 1. Clonar o repositório

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

try:
    userdata.get('GITHUB_PAT')               # secret do Colab, nunca hard-coded (o valor não é guardado nem impresso)
except Exception as erro:
    raise SystemExit('Secret GITHUB_PAT indisponível (%s). Colab: ícone de chave → "+ Add new secret" → nome '
                     'GITHUB_PAT → cole o token → ligue "Notebook access" → rode esta célula de novo.' % type(erro).__name__)
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

# O token NÃO entra na URL (ficaria gravado em .git/config): vai por GIT_ASKPASS (script 700, apagado ao
# terminar), só no ambiente do processo do git.
url = f'https://github.com/angeloalsf/{REPO_NAME}.git'

def git(*args, check=True):
    pasta = tempfile.mkdtemp(prefix='askpass-'); os.chmod(pasta, 0o700)
    script = os.path.join(pasta, 'askpass.sh')
    with open(script, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in *sername*) printf "x-access-token";; *) printf "%s" "$RECLIN_GIT_TOKEN";; esac\n')
    os.chmod(script, 0o700)
    env = {**os.environ, 'GIT_ASKPASS': script, 'GIT_TERMINAL_PROMPT': '0', 'RECLIN_GIT_TOKEN': userdata.get('GITHUB_PAT')}
    try:
        return subprocess.run(['git', '-c', 'credential.helper=', *args], env=env, check=check)
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

if not os.path.isdir(REPO):
    git('clone', url, REPO)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    git('-C', REPO, 'pull', '--rebase', '--autostash', check=False)

git('-C', REPO, 'config', 'user.name', GIT_USER_NAME)
git('-C', REPO, 'config', 'user.email', GIT_USER_EMAIL)
git('-C', REPO, 'remote', 'set-url', 'origin', url)

# Código novo do projeto: fica em REPO/codigo. Registra o commit clonado (HEAD) na proveniência de cada execução.
BRANCH = subprocess.run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO}/codigo')
from reclin.experimento import apresentacao, codigo_colab, operacoes, plano, raiz, segredos
INFO = codigo_colab.clonar_codigo(f'angeloalsf/{REPO_NAME}', BRANCH, REPO)
SHA_CODIGO = INFO['sha']
CODIGO = Path(INFO['codigo'])
raiz.ativar(CODIGO)                          # põe codigo/ no sys.path (devolve o ambiente de subprocessos, não o caminho)
print('Repo pronto em', REPO, f'| branch {BRANCH} | commit {SHA_CODIGO}')

## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Montar o Google Drive (checkpoints)

Execuções em `MyDrive/RECLin-PT/execucoes/<nome>/` — **pasta própria** de cada execução (o nome inclui a configuração). Dentro: `checkpoints/melhor_modelo/` (pesos da melhor época) e `checkpoints/ultimo.pt` (estado de retomada). O treino roda no disco local e o Drive recebe uma cópia verificada (SHA-256).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ_DRIVE = Path(f'/content/drive/MyDrive/{REPO_NAME}')
RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
RAIZ_LOCAL = Path('/content/reclin_resultados/execucoes')       # disco local do Colab
RAIZ_PESOS = Path('/content/reclin_pesos')
print('Execuções espelhadas em:', RAIZ_DRIVE)
existe = any(e['estado'] in ('interrompida', 'iniciada') for e in plano.execucoes(RAIZ_LOCAL, RAIZ_DRIVE))
print('execução interrompida existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')

## 4. Instalar dependências

**Só esta célula instala dependências** (nada é instalado ao abrir o notebook nem em outras seções). Fonte única: `codigo/requirements.txt` do repo — a mesma lista usada fora do Colab, com as versões **fixadas com `==`**.

O `pip` **não reinstala** o `torch` enquanto a imagem do Colab trouxer a versão fixada (`torch==2.11.0`; o `+cu128` da imagem é ignorado na comparação pela PEP 440), e o mesmo vale para os demais pins que já batem com a imagem, então esta célula é rápida.

⚠️ Isso vale **enquanto o Colab ficar no 2.11.0**. Se a imagem subir e esta célula começar a demorar (baixa o `torch` do PyPI, vários GB), recapture as versões e suba o pin, nunca volte para `>=`. Nunca escreva `torch==2.11.0+cu128`. Se o pip trocar um pacote já importado (por exemplo `numpy`), o Colab pede para **reiniciar a sessão**: reinicie e rode tudo de novo, do início.

In [ ]:
!pip install -q -r $REPO/codigo/requirements.txt

## 5. Conferir os splits e os pesos

As partições (`codigo/dados/particoes-v2/*.jsonl`, seed 42, nível de documento) são **versionadas e congeladas** no repo — **as mesmas de todas as estratégias**. Esta célula só **confere** (arquivos e SHA-256 contra o MANIFEST); **não baixa nem reconstrói** dados.

**Só a segunda célula desta seção baixa os pesos** (nenhuma outra seção acessa o Hub para baixar; o treino recusa se eles faltarem). Os pesos do encoder são baixados do Hub na **revisão fixada** (`REVISAO_PESOS`), com conferência dos hashes dos arquivos; o commit usado fica registrado e congelado para a fase.

In [ ]:
REVISAO_PESOS = '36b11cfdfa49876b108fdbe88eaeeb9be3b03f1c'   # commit de pucpr/biobertpt-all

def _espec():
    from reclin.estrategias import pair_aware      # importada na hora (depois da instalação da seção 4)
    # todos os pares, sem filtro lexical e sem léxico: a especificação não carrega nada além da configuração
    return operacoes.Especificacao(estrategia=pair_aware.NOME, script='treinar.py', config_cls=pair_aware.ConfigPairAware,
                                   estrategia_config=pair_aware.montagem().registro)

def _ctx(**kw):
    return apresentacao.contexto(CODIGO, RAIZ_LOCAL, RAIZ_DRIVE, RAIZ_PESOS, REVISAO_PESOS, _espec(), **kw)

def _ctx_exec(restaurar=False, **kw):
    # contexto da execução mais recente (local ou Drive); com restaurar=True traz do Drive para o disco local
    try:
        return _ctx(execucao=apresentacao.execucao_atual(_ctx(), restaurar=restaurar), **kw)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None

splits = os.path.join(REPO, 'codigo', 'dados', 'particoes-v2')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'MANIFEST.json']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    raise FileNotFoundError('Partições não encontradas em codigo/dados/particoes-v2/. Elas são versionadas e '
                            'congeladas; este notebook não baixa nem reconstrói dados. Confira a branch clonada.')

PART = apresentacao.conferir_particoes(_ctx(), sha_codigo=SHA_CODIGO)
assert PART and PART['ok'], 'Partições não conferem com o MANIFEST (veja ERRO acima).'

In [ ]:
apresentacao.avisar_revisao_pesos(_ctx())
RES_PESOS = operacoes.executar(operacoes.OP_PESOS, _ctx())
assert RES_PESOS, 'Pesos indisponíveis (veja ERRO acima).'

## 6. Token do Hugging Face Hub (Colab Secrets)

Prepara o **backup** desta execução exploratória num repositório **privado** do HF Hub (`angeloalsf/reclin-pt-pairaware-biobertpt-seed42`). É camada **extra**: o Drive continua sendo o destino principal. O envio é feito **ao final**, na seção 11, e só com a sua confirmação explícita (o `treinar.py` atual não envia nada durante o treino).

O token nunca é colado em texto plano aqui; vem dos **Secrets** do Colab, e **não é exportado** para o ambiente nesta célula.

1. Crie um Access Token do tipo **Write** em <https://huggingface.co/settings/tokens>
2. No Colab: ícone de **chave** no painel esquerdo → **+ Add new secret** → nome `HF_TOKEN` → cole o valor → ligue **Notebook access**
3. Rode a célula abaixo

Sem o secret o treino roda igual; só o backup extra fica de fora.

O repositório é o mesmo da Pair-Aware oficial; a publicação usa um **prefixo próprio da execução** (`pair_aware__<config_id>__t1`), então nada da oficial é sobrescrito.

In [ ]:
from google.colab import userdata

HF_USER = 'angeloalsf'
MODELO = 'biobertpt'

# A semente é a da fase, vem da constante do protocolo (não de nome de pasta).
SEED, _ = apresentacao.semente_da_fase()
HF_BACKUP_REPO = f'{HF_USER}/reclin-pt-pairaware-{MODELO}-seed{SEED}'

try:
    cred = segredos.obter(segredos.HUGGING_FACE)
except Exception as erro:
    cred = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)

HF_OK = False
if cred:
    try:
        quem = segredos.verificar_escrita_hf(cred)        # consulta somente leitura (whoami); nada é enviado
        HF_OK = True
        print('HF_TOKEN carregado dos Secrets do Colab; token de escrita valido (usuario %s).' % quem['usuario'])
        print('(o valor nunca e impresso, nem exportado para o ambiente nesta celula).')
        print('Backup (secao 11) ->', HF_BACKUP_REPO, '(repositorio PRIVADO)')
        print('Semente da fase:', SEED)
    except Exception as erro:
        print('HF_TOKEN presente, mas nao validou como token de escrita (%s).' % type(erro).__name__)
cred = None
if not HF_OK:
    print('BACKUP NO HF HUB DESLIGADO nesta execucao.')
    print('  1. Crie um token WRITE em https://huggingface.co/settings/tokens')
    print('  2. Colab: icone de chave (Secrets) no painel esquerdo >')
    print('     "+ Add new secret" > nome HF_TOKEN > cole o valor >')
    print('     ligue "Notebook access"')
    print('  3. Rode esta celula de novo.')
    print('  O treino roda normalmente; so o backup extra fica de fora.')

## 7. Treinar a Pair-Aware BioBERTpt-all **sem pesos de classe** (com retomada)

**7a** fixa os parâmetros e confere que a única diferença para o protocolo é `class_weight`. **7b** decide sozinha o que fazer **para esta configuração**:

* há execução **interrompida compatível** (mesma configuração, dados, pesos e identidade do código) → **retoma** de onde parou;
* já há execução **concluída** desta configuração → **não treina de novo** (nada é sobrescrito); para outra tentativa, ponha `NOVA_TENTATIVA = True`;
* senão → cria a execução exploratória (`pair_aware__<config_id>__t1`).

A decisão olha só as execuções **desta** configuração: a pasta oficial `pairaware_biobertpt_seed42` (com `class_weight=balanced`) tem outro `config_id` e nunca é retomada, reaproveitada nem sobrescrita aqui.

Se o runtime cair, rode as seções 1–6 e depois 7a e 7b de novo.

In [ ]:
# 7a — parâmetros (rode também antes das seções 8–11 numa sessão nova: elas localizam a execução por esta configuração)
PARAMETROS = {                 # protocolo (D10) com UMA mudança: class_weight
    'max_gap': 25, 'ctx_chars': 128, 'max_length': 128, 'epochs': 3, 'batch_size': 64, 'lr': 2e-5,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0,
    'class_weight': 'none',        # <<< ÚNICA diferença para a Pair-Aware oficial (era 'balanced')
    'head_dropout': None,          # None = dropout do encoder
}
MOTIVO = ('Teste exploratório: Pair-Aware linear BioBERTpt-all com class_weight=none (único parâmetro alterado) '
          'para avaliar a precisão de negation_of no DEV (auditoria de 05/11/2026)')
NOVA_TENTATIVA = False         # True só para forçar OUTRA tentativa desta mesma configuração

from reclin.experimento import persistencia, proveniencia

_ctx_teste = _ctx(parametros=PARAMETROS, motivo=MOTIVO)
_cfg_teste = operacoes.config_da_operacao(_ctx_teste.espec, PARAMETROS).como_dict()
_cfg_prot = _ctx_teste.espec.config_protocolo().como_dict()
DIFERENCAS = {k: (_cfg_prot.get(k), v) for k, v in _cfg_teste.items() if _cfg_prot.get(k) != v}
print('Diferenças em relação ao protocolo (protocolo → teste):', DIFERENCAS)
assert set(DIFERENCAS) == {'class_weight'} and DIFERENCAS['class_weight'] == ('balanced', 'none'), \
    'Este notebook só pode alterar class_weight (balanced → none); confira PARAMETROS.'

def _execucoes_desta_config():
    """As execuções (local e Drive) DESTA configuração, da tentativa mais recente para a mais antiga.
    Usa a mesma chave do plano de treino (configuração + dados + pesos + identidade do código)."""
    ctx = _ctx(parametros=PARAMETROS, motivo=MOTIVO)
    chave = operacoes._chave(ctx, operacoes.config_da_operacao(ctx.espec, PARAMETROS))
    return plano.compativeis(chave, RAIZ_LOCAL, RAIZ_DRIVE)

for e in _execucoes_desta_config():
    print(f"  já existe: {e['nome']} ({e['onde']}, {e['estado']})")

In [ ]:
# 7b — treinar / retomar / não treinar de novo
_compat = _execucoes_desta_config()
_ativas = [e for e in _compat if e['estado'] in ('interrompida', 'iniciada')]
_concluidas = [e for e in _compat if e['estado'] == 'concluida']
ctx = _ctx(parametros=PARAMETROS, motivo=MOTIVO)

if _ativas and not NOVA_TENTATIVA:
    OPERACAO = operacoes.OP_RETOMAR
    print('Há execução interrompida desta configuração:', _ativas[0]['nome'], '→ RETOMAR')
elif _concluidas and not NOVA_TENTATIVA:
    OPERACAO = None
    print('Esta configuração já foi treinada e concluída:', _concluidas[0]['nome'],
          '→ nada é treinado de novo. Siga para a seção 8. (Outra tentativa: NOVA_TENTATIVA = True.)')
else:
    OPERACAO = operacoes.OP_NOVA
    print('Nova execução EXPLORATÓRIA (class_weight=none).')

if OPERACAO is not None:
    print('\nOperação:', OPERACAO, '| natureza: EXPLORATÓRIA (fora da fase científica, não elegível ao TEST)')
    QUADRO = apresentacao.quadro_pre_treino(ctx, OPERACAO, sha_codigo=SHA_CODIGO, repositorio=f'angeloalsf/{REPO_NAME}')
    try:
        apresentacao.exigir_quadro(QUADRO, ctx, OPERACAO)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None
    RESULTADO_TREINO = operacoes.executar(OPERACAO, ctx)

## 8. Resultados (métricas + matriz de confusão)

Seleciona **a execução desta configuração** (`class_weight=none`), nunca a mais recente do encoder: a Pair-Aware oficial usa o mesmo encoder e não pode ser pega por engano. Ela fica **selecionada** para as seções 10 e 11. **Só DEV**; o TEST não é lido nem avaliado aqui. Numa sessão nova, rode as seções 1–6 e a **7a** antes desta.

In [ ]:
def _selecionar_execucao(restaurar=True):
    compat = _execucoes_desta_config()
    if not compat:
        raise SystemExit('Nenhuma execução desta configuração (class_weight=none): treine primeiro (seção 7).')
    escolhida = ([e for e in compat if e['estado'] == 'concluida'] or compat)[0]
    nome = escolhida['nome']
    if escolhida['estado'] != 'concluida':
        print(f"ATENÇÃO: {nome} está '{escolhida['estado']}' (não concluída); termine o treino (7b) antes de ler os resultados.")
    if restaurar and not (RAIZ_LOCAL / nome / proveniencia.ARQUIVO).is_file():
        print(f'{nome} só existe no Drive: restaurando para o disco local…')
        persistencia.restaurar(RAIZ_DRIVE, nome, RAIZ_LOCAL, progresso=print)
    return nome

EXECUCAO_SELECIONADA = _selecionar_execucao()     # as seções 10 e 11 enviam SÓ esta execução
CTX = _ctx(execucao=EXECUCAO_SELECIONADA, parametros=PARAMETROS)
_reg = proveniencia.ler(apresentacao.pasta_local_da_execucao(CTX))
assert (_reg['parametros_de_treinamento'] or {}).get('class_weight') == 'none', 'A execução selecionada não é a de class_weight=none!'
_base = CTX.espec.config_protocolo().como_dict()
NATUREZA = 'PROTOCOLO' if all(_reg['parametros_de_treinamento'].get(k) == _base[k] for k in CTX.espec.permitidos) else 'EXPLORATÓRIA'
print(f"Execução selecionada: {EXECUCAO_SELECIONADA} | natureza: {NATUREZA} | estado: {_reg['estado']}")
operacoes.executar(operacoes.OP_DEV, CTX)

### 8.1 Checkpoint selecionado: métricas por classe e proveniência

Critério de seleção (global, o do protocolo): **maior macro-F1 do DEV completo** (as 3 classes pesam igual; `negation_of` sozinho não decide), empate → a primeira época. As métricas abaixo vêm do `predicoes_dev.json` do checkpoint selecionado (o DEV completo, 19.064 candidatos; o que a Pair-Aware não pontua vira `no_relation`).

Por época, o `treino.json` guarda apenas `train_loss`, `dev_loss`, `dev_macro_f1` e `dev_negation_of_f1`. P/R por classe a cada época **não** são gravados (exigiria mudar o laço de treino compartilhado); só o checkpoint selecionado tem o detalhamento completo.

In [ ]:
from reclin.experimento import comparacao_dev
from reclin.execucao import predicoes as _pred

import json
PASTA = apresentacao.pasta_local_da_execucao(CTX)
_trn = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
_cfg = json.load(open(PASTA / 'config.json', encoding='utf-8'))
_dev = _pred.ler(PASTA / 'predicoes_dev.json')
_l = comparacao_dev.linha(EXECUCAO_SELECIONADA, _dev)

_ec = (_reg.get('chave_configuracao') or {}).get('estrategia_config') or {}
print('Critério de seleção :', _ec.get('selecao_melhor_epoca') or 'dev_macro_f1 no DEV completo (protocolo)')
print('Melhor época        :', _trn['melhor_epoca'], '| dev_macro_f1 =', round(_trn['melhor_dev_macro_f1'], 4))
print('\nPor época (DEV completo):')
for e in _trn['dev_history']:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in e.items()})
print(f"\nDEV completo (n = {_l['n']}) no checkpoint selecionado:  macro-F1 {_l['macro_f1']:.4f} | acurácia {_l['acuracia']:.4f} | MCC {_l['mcc']:.4f}")
for r in comparacao_dev.FOCO:
    print(f"  {r:16s} P {_l[r + '.precisao']:.4f}  R {_l[r + '.recall']:.4f}  F1 {_l[r + '.f1']:.4f}  suporte {int(_l[r + '.suporte'])}")
print('Matriz de confusão (linhas = verdadeiro, colunas = predito; negation_of, associated_with, no_relation):')
for lin in _l['matriz_confusao']: print('  ', lin)
_prov = _cfg.get('proveniencia_pesos') or {}
print('\nProveniência: seed', _reg['seed'], '| revisão dos pesos', _prov.get('commit') or _reg['pesos_pretreinados'].get('commit'),
      '| hash do manifesto dos pesos', _prov.get('hash_manifesto') or _reg['pesos_pretreinados'].get('hash_manifesto'))
print('  código:', {k: _reg['codigo'].get(k) for k in ('origem', 'revisao')}, '| identidade:', _reg['codigo'].get('identidade_global'))
print('  sha256 do config.json:', _reg.get('sha256_config'), '| natureza:', NATUREZA)
print('  configuração efetiva:', _reg['parametros_de_treinamento'])
print('  conjuntos/versões de dados:', _cfg.get('versoes_dados'))
print('  filtro lexical / léxico:', _ec.get('filtro_lexical'), '/', _ec.get('lexico'))
print('  marcadores ausentes (fallback h_cls):', (_trn.get('descricao') or {}).get('marcadores_ausentes'))

### 8.2 Comparação no DEV (universo comum)

Compara esta execução com a **Pair-Aware oficial** (`pairaware_biobertpt_seed42`, `class_weight=balanced`) e com o **Baseline** (`baseline_biobertpt_seed42`), procurando os `predicoes_dev.json` no disco local e no Drive. A comparação só é feita entre sidecars com o **mesmo `y_true`**; execuções não encontradas são listadas como tal — **nenhum valor é inventado**. Só DEV; o TEST não é lido.

In [ ]:
from reclin.experimento import comparacao_dev

NOMES_COMPARACAO = [                 # nomes reais das pastas em MyDrive/RECLin-PT/execucoes
    'baseline_biobertpt_seed42',         # Baseline, BioBERTpt-all, class_weight=balanced
    'pairaware_biobertpt_seed42',        # Pair-Aware linear OFICIAL, BioBERTpt-all, class_weight=balanced
    EXECUCAO_SELECIONADA,                # este teste: Pair-Aware linear, class_weight=none
]
RAIZES = [RAIZ_LOCAL, persistencia.raiz_das_execucoes(RAIZ_DRIVE)]
try:
    _cmp = comparacao_dev.carregar_e_comparar(RAIZES, NOMES_COMPARACAO)
    print(comparacao_dev.formatar(_cmp))
except comparacao_dev.ErroComparacao as erro:
    print('COMPARAÇÃO RECUSADA:', erro)

### 8.3 Teste × Pair-Aware oficial: diferença pareada no DEV

Diferença **teste − oficial** (mesmos 19.064 candidatos do DEV): P, R e F1 de `negation_of` lado a lado, VP/FP/FN, e a diferença de F1 por classe e do macro-F1 com **bootstrap pareado** (10.000 reamostras, seed 42) **por candidato** (o do protocolo, `reclin.avaliacao.significancia`) e **por documento** (reamostra os documentos do DEV), mais o **McNemar exato**. "Detectada" = o IC95% exclui o zero. É uma leitura **descritiva e exploratória** de uma única semente: não substitui a configuração de referência sem decisão do autor.

In [ ]:
import numpy as np
from reclin.avaliacao import metricas as _met, significancia as _sig
from reclin import tarefa as _tarefa
from reclin.execucao import predicoes as _pred
from reclin.util.io import ler_jsonl

OFICIAL = 'pairaware_biobertpt_seed42'
_c_of = comparacao_dev.localizar(RAIZES, OFICIAL)
_c_te = comparacao_dev.localizar(RAIZES, EXECUCAO_SELECIONADA)
if _c_of is None or _c_te is None:
    print('Predições não encontradas:', [n for n, c in ((OFICIAL, _c_of), (EXECUCAO_SELECIONADA, _c_te)) if c is None],
          '— nada foi calculado.')
else:
    of, te = _pred.ler(_c_of), _pred.ler(_c_te)
    assert _pred.mesmo_conjunto(of, te), 'y_true/labels diferem: as execuções não são comparáveis'
    yt, ya, yb = (np.asarray(v, dtype=np.int64) for v in (te['y_true'], te['y_pred'], of['y_pred']))

    # documento de cada candidato (mesma ordem do DEV reconstruído da partição congelada; só o DEV é lido)
    _docs = list(ler_jsonl(Path(REPO) / 'codigo' / 'dados' / 'particoes-v2' / 'dev.jsonl'))
    _cands = _tarefa.candidatos(_docs)
    assert _tarefa.y_true(_cands) == yt.tolist(), 'o DEV reconstruído não confere com o y_true das predições'
    _, grupo = np.unique([c['doc_id'] for c in _cands], return_inverse=True)

    def _cm(y):                       # matriz de confusão 3×3 (linhas = verdadeiro)
        return np.bincount(3 * yt + y, minlength=9).reshape(3, 3)
    def _prf(cm):
        tp = np.diag(cm).astype(float); pp = cm.sum(0); su = cm.sum(1)
        p = np.divide(tp, pp, out=np.zeros(3), where=pp > 0); r = np.divide(tp, su, out=np.zeros(3), where=su > 0)
        f = np.divide(2 * p * r, p + r, out=np.zeros(3), where=(p + r) > 0)
        return p, r, f
    L = list(_tarefa.LABELS)
    print(f"{'':28s}" + ''.join(f'{x:>10s}' for x in ('P', 'R', 'F1', 'VP', 'FP', 'FN')))
    for nome, y in (('oficial (balanced)', yb), ('teste (none)', ya)):
        cm = _cm(y); p, r, f = _prf(cm); k = L.index('negation_of')
        print(f'negation_of — {nome:14s}' + f'{p[k]:10.4f}{r[k]:10.4f}{f[k]:10.4f}'
              + f'{cm[k, k]:10d}{cm[:, k].sum() - cm[k, k]:10d}{cm[k].sum() - cm[k, k]:10d}')
    print()
    for nome, y in (('oficial (balanced)', yb), ('teste (none)', ya)):
        m = _met.avaliar(yt.tolist(), y.tolist())
        print(f"{nome:20s} macro-F1 {m['macro_f1']:.4f} | MCC {m['mcc']:.4f} | F1 " +
              ', '.join(f"{c} {m['por_classe'][c]['f1']:.4f}" for c in L))

    # bootstrap pareado por documento (teste − oficial)
    rng = np.random.default_rng(42); D = grupo.max() + 1
    cm_a = np.zeros((D, 9), dtype=np.int64); cm_b = np.zeros((D, 9), dtype=np.int64)
    np.add.at(cm_a, (grupo, 3 * yt + ya), 1); np.add.at(cm_b, (grupo, 3 * yt + yb), 1)
    pesos = np.stack([np.bincount(rng.integers(0, D, D), minlength=D) for _ in range(10_000)])
    def _f1_lote(cm):                 # (B, 9) → F1 por classe (B, 3)
        cm = cm.reshape(-1, 3, 3).astype(float); tp = np.diagonal(cm, axis1=1, axis2=2)
        pp = cm.sum(1); su = cm.sum(2)
        p = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0); r = np.divide(tp, su, out=np.zeros_like(tp), where=su > 0)
        return np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    d_doc = _f1_lote(pesos @ cm_a) - _f1_lote(pesos @ cm_b)
    d_doc = np.column_stack([d_doc, d_doc.mean(1)])          # + macro-F1

    print('\nDiferença teste − oficial (DEV, seed 42; IC95% por bootstrap pareado, 10.000 reamostras)')
    for j, rot in enumerate(L + ['macro_f1']):
        if rot != 'macro_f1':
            cand = _sig.bootstrap_pareado(yt, ya, yb, j, 10_000, 42)
            obs, ic_c = cand['observed_diff'], (cand['ci95_low'], cand['ci95_high'])
        else:
            obs = float(_prf(_cm(ya))[2].mean() - _prf(_cm(yb))[2].mean()); ic_c = None
        ic_d = np.percentile(d_doc[:, j], [2.5, 97.5])
        txt_c = f'cand [{ic_c[0]:+.4f}; {ic_c[1]:+.4f}]' if ic_c else 'cand —'
        lido = 'detectada' if (ic_d[0] > 0 or ic_d[1] < 0) else 'não detectada (por documento)'
        print(f"  {('F1 ' + rot) if rot != 'macro_f1' else 'macro-F1':19s} {obs:+.4f}" + f' | {txt_c} | doc [{ic_d[0]:+.4f}; {ic_d[1]:+.4f}] | {lido}')
    print('\nMcNemar exato (acerto por candidato):', _sig.mcnemar_exato(ya == yt, yb == yt))

## 9. Curvas de treino/validação

De `dev_history`: loss de treino vs. validação (overfitting) e F1 no dev por época (seleção de época). Na Pair-Aware, `dev_macro_f1` é o do DEV completo, o critério de escolha da época.

In [ ]:
import json
import matplotlib.pyplot as plt
PASTA = apresentacao.pasta_local_da_execucao(CTX)
r = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
h = r['dev_history']
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
ax[0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
ax[0].set_xlabel('época'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [e['dev_macro_f1'] for e in h], 'o-', label='dev_macro_f1')
ax[1].plot(ep, [e['dev_negation_of_f1'] for e in h], 's-', label='dev_negation_of_f1')
ax[1].set_xlabel('época'); ax[1].set_ylabel('F1'); ax[1].set_title('F1 no dev'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Salvar resultados + predições no GitHub

Forçamos o `add` (`-f`) de cinco arquivos **da execução selecionada na seção 8** (nenhuma outra), em `codigo/resultados/execucoes/<nome>/`: `config.json`, `treino.json`, `experimento.json`, `predicoes_dev.json` (predições do dev na melhor época) e `metricas.json` (**só DEV**). Depois `commit`, `pull --rebase --autostash` e `push`, como no legado.

Nunca vão para o GitHub: `ultimo.pt`, checkpoints, pesos, logs nem artefatos do TEST. O envio exige execução concluída, `metricas.json` sem a chave `test` e predições sem texto clínico, e vai para uma **branch própria** (`main` e `master` são recusadas; o merge para a main é manual, fora do notebook).

**Nada é enviado até você mudar `PUBLICAR_NO_GITHUB` para `True`.** Antes, a célula mostra exatamente o que seria enviado.

In [ ]:
from reclin.experimento import versionamento

PUBLICAR_NO_GITHUB = False          # True = confirmo enviar estes arquivos do DEV ao GitHub
BRANCH_RESULTADOS = 'resultados-colab'   # fixa (main e master são recusadas)

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA)
PASTA = apresentacao.pasta_local_da_execucao(CTX)
print('Execução a enviar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
try:
    versionamento.conferir(PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO)
    if not PUBLICAR_NO_GITHUB:
        print('\nNada foi enviado. Confira acima; para enviar, ponha PUBLICAR_NO_GITHUB = True e rode de novo.')
    else:
        _cred = segredos.Credencial('GITHUB_PAT', userdata.get('GITHUB_PAT'), 'Colab Secrets')
        try:
            RES_GIT = versionamento.salvar(
                PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO,
                clone='/content/reclin_versionamento', nome_git=GIT_USER_NAME, email_git=GIT_USER_EMAIL, credencial=_cred)
            print('Push concluído (se houve mudanças).')
        except versionamento.ErroVersionamento as erro:
            print('ERRO:', segredos.limpar(str(erro), _cred))
        finally:
            _cred = None
except versionamento.ErroVersionamento as erro:
    print('ERRO:', erro)

## 11. Backup privado do melhor modelo no HF Hub (confirmação explícita)

Publica a execução selecionada na seção 8 (`melhor_modelo/` + manifesto com SHA-256, **sem** `ultimo.pt`) no repositório **privado** `HF_BACKUP_REPO` (seção 6), num prefixo próprio da execução; nada de execuções anteriores é sobrescrito. Use também como **rede de segurança** se a rede cair no meio de um envio: rode de novo.

A célula primeiro mostra o plano (o que seria enviado) e **só envia** com `PUBLICAR_NO_HF = True`. O `HF_TOKEN` é lido dos Secrets **só durante o envio** e removido do ambiente ao terminar.

In [ ]:
PUBLICAR_NO_HF = False              # True = confirmo publicar a execução no repositório PRIVADO HF_BACKUP_REPO

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA, repositorio_hf=HF_BACKUP_REPO)
CTX.confirmacao = operacoes.frase_de_publicacao(CTX.execucao, HF_BACKUP_REPO) if PUBLICAR_NO_HF else ''
print('Execução a publicar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
operacoes.executar(operacoes.OP_PUBLICACAO, CTX)             # plano: nada é enviado
if not PUBLICAR_NO_HF:
    print('\nNada foi enviado. Para publicar, ponha PUBLICAR_NO_HF = True e rode de novo.')
else:
    try:
        with apresentacao.segredo_no_ambiente(segredos.HUGGING_FACE):
            RES_PUBLICACAO = operacoes.executar(operacoes.OP_PUBLICAR, CTX)
        print('best_model em https://huggingface.co/' + HF_BACKUP_REPO)
    except segredos.ErroSegredo as erro:
        print('ERRO:', erro)
        print(apresentacao.instrucoes_secret(segredos.HUGGING_FACE))